# Differentially expressed genes in neighborhoods
This notebook will perform differential abundance analysis, using the approach generated by Andreas Fønss Møller in the hackathon

In [1]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_milo
# python -m ipykernel install --user --name scrna_cartography_milo --display-name "milo"

#### Load libraries

In [2]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pathlib import Path     # Object-oriented filesystem paths
from pyhere import here      # Reproducible project paths

# Single-cell data handling
import anndata as ad            # Core data structure for single-cell data
import scanpy as sc

# milo
import pertpy as pt
milo = pt.tl.Milo()
import mudata as md
from mudata import MuData

# Parallel processing
from joblib import Parallel, delayed, parallel_backend

# dataframes
import pandas as pd
import numpy as np

# plotting
import matplotlib.pyplot as plt 

import random
from sklearn.metrics.pairwise import euclidean_distances
from sklearn_ann.kneighbors.annoy import AnnoyTransformer 

# Custom modules and functions
sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import diff_genes as dg
import misc as mi
import dg_milo as dm

/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Set up

In [3]:
# Create directories
mi.create_directories(dir_path = str(here('data/milo/files')))
mi.create_directories(dir_path = str(here('data/milo/plots')))
mi.create_directories(dir_path = str(here('data/milo/objects')))

/work/islet_cartography_scrna/data/milo/files Directory already exists!
/work/islet_cartography_scrna/data/milo/plots Directory already exists!
/work/islet_cartography_scrna/data/milo/objects Directory already exists!


In [4]:
# Paths
base_dir = str(here('data/milo/'))
plot_dir = os.path.join(base_dir, 'plot') 
files_dir = os.path.join(base_dir, 'files') 
objects_dir = os.path.join(base_dir, 'objects') 
anndata_dir = str(here('data/anndata/'))

#### Load

In [5]:
adata = ad.read_h5ad(os.path.join(anndata_dir, "AH_combined.h5ad"))

##### T2DM

In [6]:
res_t2d = pd.read_csv(os.path.join(files_dir, "t2d_vs_nd.csv"), index_col=0) # diff abundance results
res_t2d.reset_index(drop=True, inplace = True)
df = pd.read_csv(os.path.join(files_dir, "cells_in_neighborhood.csv"), index_col=0) # cells in neighborhoods
df.reset_index(drop=True, inplace = True)

In [7]:
# Combine results
df['index_cell'] = df['neighborhood']
df = df.merge(res_t2d, how = "left", on = "index_cell")

In [8]:
# Add direction of regulation
df.loc[(df["SpatialFDR"] <= 0.1) & (df["logFC"] > 0), 'direction'] = 'up'
df.loc[(df["SpatialFDR"] <= 0.1) & (df["logFC"] < 0), 'direction'] = 'down'
df.loc[df["direction"].isna(), 'direction'] = "no_change"
# Add a key for differential expression
df['diff_key'] = df['nhood_annotation'] + '_' + df['direction']
# Set index
df = df.set_index('cell')

In [ ]:
nhood_anno = 'nhood_annotation'
cluster_key = 'diff_key'
sample_key = 'ic_id_platform_adjusted_sample'
prefix  = 't2d_vs_nd'
for anno_id in df[nhood_anno].unique():
    print(f'Processing {anno_id}')
    
    # Subset df to only contain annotation - drop duplicated cells and set cell as index (for merge)
    df_sub = df.loc[df[nhood_anno] == anno_id].copy()

    # Remove duplicated cells 
    df_sub = (
        df_sub[[cluster_key]]
        .loc[~df_sub.index.duplicated(keep='first')]
    )
    
    # Get cluster ids to test
    cluster_ids = [
        x for x in df_sub[cluster_key].unique()
        if isinstance(x, str) and (x.endswith("_up") or x.endswith("_down"))
    ]

    if len(cluster_ids) == 0:
        print(f"No differentally abundance neighborshoods for {anno_id} skipping...")
        continue
        
    # Create clustering columns
    for cluster_id in cluster_ids:
        print(f'Differential expression for {cluster_id}')
        col_name = f"{cluster_id}_vs_other"
        df_sub[col_name] = df_sub[cluster_key].apply(
            lambda x: cluster_id if x == cluster_id else 'other'
        )

         # subset ann data object to neighborhoods of interest
        subset = adata[adata.obs_names.isin(df_sub.index.unique())].copy()

        subset.obs = subset.obs.merge(df_sub, how = "left", left_index = True, right_index = True)
        
        dds = dg.prepare_pseudobulk_deseq_analysis(
            ad=subset,
            sample_key=sample_key,
            cluster_key=col_name,
            no_subset=True,
            n_cells = 25,
            design=f'~ {sample_key} + {col_name}',
            layer="counts",
            func='sum',
            return_all = False,
            workers=30)

        # Define comparison
        c1 = cluster_id
        c2 = 'other'

        # Wald test
        res = dg.diff_genes_two_clusters(dds_obj = dds, 
                              cluster_index = col_name, 
                              cluster_1 = c1, 
                              cluster_2 = c2, 
                              workers = 30)
        # Save results
        res.to_csv(os.path.join(files_dir, f"{prefix}_deg_{cluster_id}_vs_other.csv"), index_label= "gene_symbol")

        # Save normalized counts
        norm_df = pd.DataFrame(
        dds.layers["normed_counts"],
        index=dds.obs_names,
        columns=dds.var_names)

        norm_df.transpose().to_csv(os.path.join(files_dir, f"{prefix}_norm_counts_{cluster_id}_vs_other.csv"), index_label= "gene_symbol")

Processing beta
Differential expression for beta_down


Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 1.08 seconds.

Fitting dispersions...
... done in 125.22 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.69 seconds.

Fitting MAP dispersions...
... done in 93.53 seconds.

Fitting LFCs...
... done in 217.03 seconds.

Calculating cook's distance...
... done in 8.11 seconds.

Replacing 0 outlier genes.



Differential expression for beta_up


...

... done in 0.78 seconds.

Fitting dispersions...
... done in 138.10 seconds.

Fitting dispersion trend curve...
/work/islet_cartography_scrna/scrna_cartography_milo/lib/python3.13/site-packages/pydeseq2/dds.py:820: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.69 seconds.

Fitting MAP dispersions...
... done in 91.48 seconds.

Fitting LFCs...
... done in 232.00 seconds.

Calculating cook's distance...
... done in 0.96 seconds.

Replacing 0 outlier genes.



Processing mixed
Differential expression for mixed_down


Fitting size factors...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.70 seconds.

